# The Digital Border — Country-Level Extensive Margin (Melitz–Chaney)

**Standalone companion to `gravity_thesis.ipynb`.** This notebook re-implements the *country-level extensive margin* using the heterogeneous-firms framework in Allen & Arkolakis, *Trade with Heterogeneous Firms* (the Melitz 2003 / Chaney 2008 model). It estimates whether importer data-localization (DLR) reduces the **number of service varieties** an exporter sells into a destination — the empirical analogue of the Melitz **mass of exporters** $M_{ij}$.

## What the paper says
A firm from origin $i$ exports to destination $j$ only if its productivity clears a destination-specific cutoff (eq. 10):
$$\varphi^{*}_{ij} = \Big[\tfrac{\sigma f_{ij}}{Y_j P_j^{\sigma-1}}\big(\tfrac{\sigma}{\sigma-1}w_i\tau_{ij}\big)^{\sigma-1}\Big]^{\frac{1}{\sigma-1}}.$$
The **extensive margin** is the mass of firms that clear it (eq. 12), which under a Pareto productivity distribution $G_i(\varphi)=1-\varphi^{-\theta}$ becomes
$$M_{ij} = \big[1-G_i(\varphi^{*}_{ij})\big]M_i = (\varphi^{*}_{ij})^{-\theta}\,M_i .$$
The trade-cost elasticity decomposes (Section 4.1) into $\theta = \underbrace{(\sigma-1)}_{\text{intensive}} + \underbrace{(\theta-\sigma+1)}_{\text{extensive}}$. A DLR raises the cost of serving $j$ (a rise in $f_{ij}$/$\tau_{ij}$) $\Rightarrow$ higher $\varphi^{*}_{ij}$ $\Rightarrow$ lower $M_{ij}$: **fewer varieties traded**.

## Operationalization
The paper is *theoretical* — it has no estimator. Its $M_{ij}$ maps to the standard empirical **count of traded varieties** (Hummels–Klenow 2005; Chaney 2008; Bernard et al. 2007). We build it at the country-pair level $(i,j,t)$ two ways:
- `n_bpm6` — number of distinct **BPM6 service categories** with positive trade (0–14). *Agreed primary measure.*
- `n_cmd` — number of distinct **Comtrade `cmdCode`s** with positive trade (finer, up to ~121). *Closer to the model's "varieties"; reported alongside.*

## Treatment & identification
Collapsing categories makes the treatment importer-level: `dlr_any_jt` (any active Pillar-6 restriction) primary; `dlr_count_jt` (count of restricted categories) as a dose-response. Because an importer-time fixed effect would mechanically absorb an importer-level policy, we follow the agreed design: **exporter-time + pair fixed effects**, with **importer GDP and population** standing in for the inadmissible inward multilateral-resistance term. Estimator: **PPML** (`fixest::fepois`), cluster `pair_id`.

## Honest caveats (flagged for the thesis / supervisor)
1. The paper gives a *theory* of the extensive margin, not a regression; the variety-count is a justified bridge, not a quote.
2. Varieties are coarse at BPM6 (14); the `cmdCode` count (322) materially mitigates this.
3. No full inward multilateral resistance — Comtrade has no intra-national services flows for the Heid–Larch–Yotov (2021) domestic-trade fix, so importer-year shocks are only partly controlled (GDP/pop proxy).
4. $\theta$ and $\sigma$ are not separately estimated; the DLR coefficient is the *reduced-form* extensive-margin semi-elasticity, interpreted through the $\theta-\sigma+1$ channel.

## 1 — Setup & libraries

In [ ]:
# --- Required packages --------------------------------------------------------
required_packages <- c(
  "dplyr", "tidyr", "stringr", "purrr", "readr",   # data manipulation
  "fixest",                                         # PPML / fixed-effect estimation
  "ggplot2", "scales", "viridis",                   # visualisation
  "writexl"                                         # results export
)
for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE))
    install.packages(pkg, repos = "https://cloud.r-project.org")
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}
options(jupyter.rich_display = FALSE)
options(scipen = 999)

# --- Path resolution (relative to the repository; run from notebooks/) ------
candidate_bases <- c(
  "../data",   # when run from notebooks/
  "data"       # when run from the repository root
)
BASE_DIR <- candidate_bases[which(dir.exists(candidate_bases))[1]]
if (is.na(BASE_DIR)) stop("Could not locate the data/ folder — run from notebooks/ or the repo root.")

MERGED_CSV  <- file.path(BASE_DIR, "final_dataset", "comtrade_dti_final.csv")

# Output location — probe writability at runtime and fall back if the preferred
# gravity_results/ folder is not writable here (e.g. a read-only or synced folder).
# Evaluated in your R session, so it reflects real permissions.
pick_writable <- function(cands) {
  for (d in cands) {
    ok <- tryCatch({
      dir.create(d, recursive = TRUE, showWarnings = FALSE)
      tf <- file.path(d, ".wtest"); suppressWarnings(writeLines("x", tf))
      suppressWarnings(try(unlink(tf), silent = TRUE)); TRUE
    }, error = function(e) FALSE)
    if (ok) return(d)
  }
  stop("No writable output directory found - edit the candidate list.")
}
OUTPUT_DIR <- pick_writable(c(
  file.path(BASE_DIR, "..", "outputs", "gravity_results"),  # preferred (matches headline notebook)
  file.path(BASE_DIR, "..", "outputs", "em_country_outputs"),  # fallback
  path.expand("~/em_country_outputs"),        # home-dir fallback
  file.path(tempdir(), "em_country_outputs")  # guaranteed last resort
))
TABLES_DIR  <- file.path(OUTPUT_DIR, "tables")
FIGURES_DIR <- file.path(OUTPUT_DIR, "figures")
for (d in c(TABLES_DIR, FIGURES_DIR))
  if (!dir.exists(d)) dir.create(d, recursive = TRUE, showWarnings = FALSE)
cat(sprintf("Outputs -> %s\n", OUTPUT_DIR))

# --- Global parameters & helpers ---------------------------------------------
# Row-level DLR = any active Pillar-6 (cross-border data) sub-pillar 6.1-6.5
# "negative effect" flag. Identical definition to gravity_thesis.ipynb's dlr_any.
P6_NEG_COLS <- paste0("p6_", 1:5, "_neg_effect")

# Significance stars — thesis convention.
sig_stars <- function(p) ifelse(is.na(p), "",
  ifelse(p < 0.01, "***", ifelse(p < 0.05, "**", ifelse(p < 0.10, "*", ""))))

# ggplot publication theme (identical to gravity_thesis.ipynb).
theme_thesis <- function(base_size = 11) {
  theme_minimal(base_size = base_size) +
    theme(plot.title    = element_text(face = "bold", size = base_size + 2, margin = margin(b = 8)),
          plot.subtitle = element_text(colour = "grey30", margin = margin(b = 12)),
          panel.grid.minor = element_blank(),
          panel.grid.major = element_line(colour = "grey92"),
          axis.title.x  = element_text(face = "bold", margin = margin(t = 8)),
          axis.title.y  = element_text(face = "bold", margin = margin(r = 8)),
          legend.position = "bottom", strip.text = element_text(face = "bold"))
}
theme_set(theme_thesis())
cat(sprintf("Setup complete. BASE_DIR = %s\n", BASE_DIR))

## 2 — Load data
Same merged DTI × Comtrade source as the headline notebook. Re-uses the in-memory object if `gravity_thesis.ipynb` has already been run in this session.

In [ ]:
# --- 2.1 Load merged dataset --------------------------------------------------
if (exists("full_df_merged") && is.data.frame(full_df_merged) && nrow(full_df_merged) > 0) {
  df_raw <- full_df_merged
  cat("Using in-memory `full_df_merged`.\n")
} else {
  cat(sprintf("Reading: %s\n", MERGED_CSV))
  df_raw <- readr::read_csv(MERGED_CSV, show_col_types = FALSE, progress = FALSE,
                            guess_max = 50000)
}
cat(sprintf("Loaded: %s rows x %d cols\n", format(nrow(df_raw), big.mark = ","), ncol(df_raw)))

# Drop Comtrade aggregate rows (NA bpm6_category) — same rule as the headline:
# they would double-count the cmdCode-level detail.
df_raw <- df_raw %>% filter(!is.na(bpm6_category))
cat(sprintf("After dropping NA-bpm6 aggregate rows: %s rows\n",
            format(nrow(df_raw), big.mark = ",")))

## 3 — Data preparation
Mirrors `gravity_thesis.ipynb` (cell 1.5): the merged file is import flows (`M`) only, so **importer = reporter**, **exporter = partner**. Negative trade (accounting revisions) floored to zero; self-trade dropped. Row-level `dlr_any` reuses the headline definition.

In [ ]:
# --- 3.1 Row-level preparation ------------------------------------------------
df <- df_raw %>%
  mutate(
    importer = if_else(flowCode == "M", reporterDesc, partnerDesc),
    exporter = if_else(flowCode == "M", partnerDesc,  reporterDesc),
    year     = as.integer(refYear),
    trade    = as.numeric(primaryValue)
  )
df$trade[is.na(df$trade)] <- 0
df$trade[df$trade < 0]    <- 0   # accounting revisions -> 0 (Anderson & Yotov 2016)
df <- df %>% filter(importer != exporter, !is.na(importer), !is.na(exporter))

# Row-level DLR: any active P6.1-6.5 negative-effect regulation on this
# (importer x BPM6-category x year) cell. Identical to headline dlr_any.
stopifnot(all(P6_NEG_COLS %in% names(df)))
df$dlr_any <- as.integer(rowSums(df[, P6_NEG_COLS] == 1, na.rm = TRUE) > 0)

cat(sprintf("Row-level panel: %s rows | importers %d | exporters %d | %d-%d\n",
            format(nrow(df), big.mark = ","), n_distinct(df$importer),
            n_distinct(df$exporter), min(df$year), max(df$year)))
cat(sprintf("Grain — distinct BPM6 categories: %d | distinct cmdCodes: %d\n",
            n_distinct(df$bpm6_category), n_distinct(df$cmdCode)))

## 4 — Importer macro controls (World Bank WDI)
The agreed identification replaces the inadmissible importer-time fixed effect with **observable importer size** (GDP, population). Comtrade names are mapped to ISO3, then GDP/population are fetched from the WDI API. If the API is unreachable the notebook degrades gracefully to an FE-only headline (`has_controls <- FALSE`).

In [ ]:
# --- 4.1 Country-name -> ISO3, then WDI GDP & population ----------------------
has_controls <- FALSE
if (!requireNamespace("countrycode", quietly = TRUE))
  install.packages("countrycode", repos = "https://cloud.r-project.org")
suppressPackageStartupMessages(library(countrycode))

# Manual overrides for Comtrade names countrycode resolves poorly (subset of the
# headline notebook's crosswalk — importers are the 56 reporters, all covered).
manual_iso <- c(
  "REP. OF KOREA"="KOR", "DEM. PEOPLES REP. OF KOREA"="PRK",
  "DEM. REP. OF THE CONGO"="COD", "TURKIYE"="TUR", "TURKEY"="TUR", "CZECHIA"="CZE",
  "VIET NAM"="VNM", "VIETNAM"="VNM", "USA"="USA", "RUSSIAN FEDERATION"="RUS",
  "CHINA, HONG KONG SAR"="HKG", "CHINA, MACAO SAR"="MAC",
  "CHINA, TAIWAN PROVINCE OF"="TWN", "BOLIVIA (PLURINATIONAL STATE OF)"="BOL",
  "VENEZUELA (BOLIVARIAN REPUBLIC OF)"="VEN", "IRAN (ISLAMIC REPUBLIC OF)"="IRN",
  "REP. OF MOLDOVA"="MDA", "STATE OF PALESTINE"="PSE", "LAO PEOPLE'S DEM. REP."="LAO",
  "BRUNEI DARUSSALAM"="BRN", "CABO VERDE"="CPV", "ESWATINI"="SWZ",
  "NORTH MACEDONIA"="MKD", "REP. OF NORTH MACEDONIA"="MKD", "SYRIAN ARAB REPUBLIC"="SYR",
  "UNITED REP. OF TANZANIA"="TZA", "UNITED KINGDOM"="GBR", "UNITED ARAB EMIRATES"="ARE",
  "REP. OF SERBIA"="SRB", "OTHER ASIA, NES"=NA, "OTHER AFRICA, NES"=NA,
  "FREE ZONES"=NA, "BUNKERS"=NA, "OTHER EUROPE, NES"=NA, "AREAS, NES"=NA,
  "SPECIAL CATEGORIES"=NA, "EUROPEAN UNION"=NA
)
map_to_iso3 <- function(nm) {
  iso  <- unname(manual_iso[nm])
  need <- is.na(iso) & !(nm %in% names(manual_iso))
  if (any(need))
    iso[need] <- suppressWarnings(countrycode(nm[need], "country.name", "iso3c", warn = FALSE))
  iso
}
df$iso_d <- map_to_iso3(df$importer)   # destination = importer
df$iso_o <- map_to_iso3(df$exporter)   # origin = exporter

if (!requireNamespace("WDI", quietly = TRUE))
  install.packages("WDI", repos = "https://cloud.r-project.org")
suppressPackageStartupMessages(library(WDI))
wdi_raw <- tryCatch(
  WDI(country = "all", indicator = c(gdp = "NY.GDP.MKTP.CD", pop = "SP.POP.TOTL"),
      start = 2000, end = max(df$year, na.rm = TRUE)),
  error = function(e) { cat(sprintf("WDI API failed: %s\n", e$message)); NULL })

if (!is.null(wdi_raw)) {
  wdi <- wdi_raw %>% filter(!is.na(iso3c), iso3c != "") %>%
    transmute(iso3 = iso3c, year = as.integer(year),
              gdp = as.numeric(gdp), pop = as.numeric(pop))
  df <- df %>%
    left_join(wdi %>% rename(iso_d = iso3, gdp_d = gdp, pop_d = pop), by = c("iso_d", "year"))
  has_controls <- TRUE
  cat(sprintf("WDI merged. Importer GDP coverage: %.1f%% of rows.\n",
              100 * mean(!is.na(df$gdp_d))))
} else {
  cat("Proceeding without importer controls (headline -> FE-only).\n")
}

## 5 — Construct the country-level extensive-margin variables
Two steps: (i) build the importer-year DLR treatment; (ii) collapse the row-level panel to one observation per importer–exporter–year, counting distinct traded varieties. Pair- and importer-level controls are constant within $(i,j,t)$ across categories, so `first()` is exact.

In [ ]:
# --- 5.1 Importer-year DLR treatment ------------------------------------------
#   dlr_any_jt   = 1 if importer j has ANY active P6 regulation in year t (primary)
#   dlr_count_jt = number of distinct BPM6 categories restricted (dose-response)
imp_treat <- df %>%
  group_by(importer, year) %>%
  summarise(
    dlr_any_jt   = as.integer(any(dlr_any == 1)),
    dlr_count_jt = n_distinct(bpm6_category[dlr_any == 1]),
    .groups = "drop"
  )

# --- 5.2 Collapse to the country-pair extensive margin (i, j, t) --------------
ctrl_cols <- intersect(c("gdp_d", "pop_d"), names(df))   # importer size only
em <- df %>%
  group_by(importer, exporter, year) %>%
  summarise(
    n_bpm6      = n_distinct(bpm6_category[trade > 0]),   # 0-14 (agreed primary)
    n_cmd       = n_distinct(cmdCode[trade > 0]),         # finer varieties
    any_trade   = as.integer(any(trade > 0)),
    total_trade = sum(trade),
    across(all_of(ctrl_cols), first),
    .groups = "drop"
  ) %>%
  left_join(imp_treat, by = c("importer", "year")) %>%
  mutate(
    exp_time = paste(exporter, year, sep = "_"),          # outward MR (exporter-year)
    pair     = paste(importer, exporter, sep = "-"),      # directed pair FE
    pair_id  = if_else(importer < exporter,               # symmetric id for clustering
                       paste(importer, exporter, sep = "-"),
                       paste(exporter, importer, sep = "-"))
  )
if (has_controls) em <- em %>% mutate(
  log_gdp_d = ifelse(!is.na(gdp_d) & gdp_d > 0, log(gdp_d), NA_real_),
  log_pop_d = ifelse(!is.na(pop_d) & pop_d > 0, log(pop_d), NA_real_)
)

cat(sprintf("Extensive-margin panel: %s (i,j,t) rows | %d pairs | %d importers | %d exporters\n",
            format(nrow(em), big.mark = ","), n_distinct(em$pair),
            n_distinct(em$importer), n_distinct(em$exporter)))
cat(sprintf("  mean n_bpm6 = %.2f (max %d) | mean n_cmd = %.2f (max %d)\n",
            mean(em$n_bpm6), max(em$n_bpm6), mean(em$n_cmd), max(em$n_cmd)))
cat(sprintf("  importer-years with DLR active: %.1f%%\n", 100 * mean(em$dlr_any_jt)))

# Identifying variation under exp_time + pair FE: within-pair over-time switches.
sw <- em %>% group_by(pair) %>% summarise(v = n_distinct(dlr_any_jt), .groups = "drop")
cat(sprintf("  within-pair DLR switchers: %d of %d pairs (%.1f%%)\n",
            sum(sw$v > 1), nrow(sw), 100 * mean(sw$v > 1)))

## 6 — Estimation
PPML (`fepois`) on the variety counts with exporter-time + pair fixed effects and importer GDP/population controls. Structural reading: $\ln \mathbb{E}[N_{ij}]$ tracks $\ln M_{ij} = -\theta\ln\varphi^{*}_{ij}+\text{const}$, so the DLR coefficient is the extensive-margin ($\theta-\sigma+1$) response to a data-localization cost shock. The headline %-effect is $(\exp\hat\beta-1)\times100$.

| Model | Outcome | Treatment | Role |
|---|---|---|---|
| E1 | BPM6 count | binary | **headline** |
| E2 | cmdCode count | binary | finer varieties |
| E3 | BPM6 count | count of restricted cats | dose-response |
| E4 | cmdCode count | count | dose-response (finer) |
| E5 | BPM6 count | binary, **no importer controls** | FE-only robustness |
| E6 | any-trade (0/1) | binary | bridge to old Section 3h (LPM) |

In [ ]:
# --- 6.1 Estimate all specifications ------------------------------------------
models   <- list()
fe_rhs   <- "| exp_time + pair"
ctrl_rhs <- if (has_controls) " + log_gdp_d + log_pop_d" else ""

# E1 headline: BPM6 variety count, binary DLR
models[["E1_bpm6_any"]] <- fepois(
  as.formula(paste0("n_bpm6 ~ dlr_any_jt", ctrl_rhs, " ", fe_rhs)),
  data = em, cluster = ~ pair_id)
# E2 finer varieties: cmdCode count, binary DLR
models[["E2_cmd_any"]] <- fepois(
  as.formula(paste0("n_cmd ~ dlr_any_jt", ctrl_rhs, " ", fe_rhs)),
  data = em, cluster = ~ pair_id)
# E3 / E4 dose-response: count of restricted categories
models[["E3_bpm6_count"]] <- fepois(
  as.formula(paste0("n_bpm6 ~ dlr_count_jt", ctrl_rhs, " ", fe_rhs)),
  data = em, cluster = ~ pair_id)
models[["E4_cmd_count"]] <- fepois(
  as.formula(paste0("n_cmd ~ dlr_count_jt", ctrl_rhs, " ", fe_rhs)),
  data = em, cluster = ~ pair_id)
# E5 FE-only robustness (drop importer controls)
models[["E5_bpm6_FEonly"]] <- fepois(
  n_bpm6 ~ dlr_any_jt | exp_time + pair, data = em, cluster = ~ pair_id)
# E6 binary "any trade" country-pair margin (LPM) — bridges to old Section 3h
models[["E6_anytrade_LPM"]] <- feols(
  as.formula(paste0("any_trade ~ dlr_any_jt", ctrl_rhs, " ", fe_rhs)),
  data = em, cluster = ~ pair_id)

# Estimator label per model (used downstream; avoids fragile method sniffing).
est_type    <- c(E1_bpm6_any="PPML", E2_cmd_any="PPML", E3_bpm6_count="PPML",
                 E4_cmd_count="PPML", E5_bpm6_FEonly="PPML", E6_anytrade_LPM="LPM")
outcome_lbl <- c(E1_bpm6_any="BPM6 count (0-14)", E2_cmd_any="cmdCode count",
                 E3_bpm6_count="BPM6 count", E4_cmd_count="cmdCode count",
                 E5_bpm6_FEonly="BPM6 count", E6_anytrade_LPM="Any trade (0/1)")

for (nm in names(models)) {
  cat(sprintf("\n===================== %s =====================\n", nm))
  print(summary(models[[nm]]))
}
cat("\nNote: N may differ across models due to listwise deletion on importer controls.\n")

## 7 — Diagnostics
Count-appropriate checks: outcome distribution and zero share (PPML handles zeros natively), the Santos Silva & Tenreyro (2006) pseudo-$R^2$ (squared correlation of observed vs fitted, since conventional $R^2$ is undefined for Poisson MLE), a parity check against the verified Python mirror, and the raw dose–count correlation.

In [ ]:
# --- 7.1 Diagnostics ----------------------------------------------------------
cat("Distribution of active-variety counts:\n")
print(summary(em[, c("n_bpm6", "n_cmd")]))
cat(sprintf("Share of (i,j,t) with zero active categories: %.2f%%\n",
            100 * mean(em$n_bpm6 == 0)))

cat("\nPPML pseudo-R2 (squared corr. of observed vs fitted; Santos Silva & Tenreyro 2006):\n")
for (nm in c("E1_bpm6_any","E2_cmd_any","E3_bpm6_count","E4_cmd_count","E5_bpm6_FEonly"))
  cat(sprintf("  %-16s cor2 = %.4f\n", nm, fixest::r2(models[[nm]], type = "cor2")))

# Parity check vs the verified mirror used during development.
cat(sprintf("\nParity vs verified mirror: pairs=%d (exp ~4,037), mean n_bpm6=%.2f (exp ~6.79), mean n_cmd=%.2f (exp ~25.35)\n",
            n_distinct(em$pair), mean(em$n_bpm6), mean(em$n_cmd)))
cat(sprintf("Raw corr(dlr_count_jt, n_bpm6) = %.3f | corr(dlr_count_jt, n_cmd) = %.3f\n",
            cor(em$dlr_count_jt, em$n_bpm6), cor(em$dlr_count_jt, em$n_cmd)))
cat("(Raw correlations are descriptive; within-FE identifying variation is what the models use.)\n")

## 8 — Results

In [ ]:
# --- 8.1 Consolidated DLR-coefficient table ----------------------------------
coef_tbl <- purrr::map_dfr(names(models), function(nm) {
  m   <- models[[nm]]
  ct  <- as.data.frame(summary(m)$coeftable)
  key <- grep("^dlr_", rownames(ct), value = TRUE)[1]
  b <- ct[key, 1]; se <- ct[key, 2]; p <- ct[key, 4]
  ppml <- est_type[[nm]] == "PPML"
  data.frame(
    model       = nm,
    outcome     = outcome_lbl[[nm]],
    estimator   = est_type[[nm]],
    treatment   = key,
    estimate    = round(b, 4),
    std_error   = round(se, 4),
    p_value     = round(p, 4),
    stars       = sig_stars(p),
    effect      = ifelse(ppml, round((exp(b) - 1) * 100, 2), round(b * 100, 2)),
    effect_unit = ifelse(ppml, "% change in variety count", "pp change in P(any trade)"),
    ci_low      = round(b - 1.96 * se, 4),
    ci_high     = round(b + 1.96 * se, 4),
    n_obs       = as.integer(nobs(m)),
    stringsAsFactors = FALSE)
})
cat("DLR effect on the country-level extensive margin:\n\n")
print(coef_tbl, row.names = FALSE)

# --- 8.2 Figure: extensive margin over time, by importer DLR status -----------
fig_df <- em %>%
  mutate(dlr = factor(dlr_any_jt, levels = c(0, 1),
                      labels = c("No DLR", "DLR active"))) %>%
  group_by(year, dlr) %>% summarise(mean_n = mean(n_bpm6), .groups = "drop")
p_trend <- ggplot(fig_df, aes(year, mean_n, colour = dlr)) +
  geom_line(linewidth = 1) + geom_point(size = 1.6) +
  scale_colour_manual(values = c("No DLR" = "grey50", "DLR active" = "#B2182B")) +
  labs(title = "Country-pair extensive margin over time",
       subtitle = "Mean number of active BPM6 service categories per importer-exporter pair",
       x = NULL, y = "Mean active categories", colour = NULL)
# Render with a headless-safe device, then embed inline via IRdisplay. On
# macOS the quartz device fails in headless kernels for BOTH file output and
# the kernel's inline plot capture, so we render through ragg (no system libs)
# or cairo, then push the saved PNG into the cell output explicitly.
if (!requireNamespace("ragg", quietly = TRUE))
  try(install.packages("ragg", repos = "https://cloud.r-project.org"), silent = TRUE)
if (!dir.exists(FIGURES_DIR)) dir.create(FIGURES_DIR, recursive = TRUE, showWarnings = FALSE)
fig_path <- file.path(FIGURES_DIR, "extensive_margin_country_trend.png")
open_dev <- function() {
  if (requireNamespace("ragg", quietly = TRUE))
    ragg::agg_png(fig_path, width = 8, height = 4.5, units = "in", res = 300)
  else
    tryCatch(grDevices::png(fig_path, 8, 4.5, units = "in", res = 300, type = "cairo"),
             error = function(e) grDevices::png(fig_path, 8, 4.5, units = "in", res = 300))
}
saved <- tryCatch({ open_dev(); print(p_trend); grDevices::dev.off(); TRUE },
                  error = function(e) { message("Figure not rendered: ", conditionMessage(e)); FALSE })
if (saved && requireNamespace("IRdisplay", quietly = TRUE)) {
  IRdisplay::display_png(file = fig_path)
  cat(sprintf("\nFigure saved & displayed: %s\n", fig_path))
} else print(p_trend)

## 9 — Export outputs
A self-contained workbook (`extensive_margin_country_results.xlsx`) plus a coefficients CSV and serialized models/panel, written to the same `gravity_results/` tree as the headline notebook (distinct filenames, so nothing is overwritten).

In [ ]:
# --- 9.1 Export ---------------------------------------------------------------
diag_tbl <- data.frame(
  diagnostic = c("(i,j,t) observations", "distinct importer-exporter pairs",
                 "mean active BPM6 categories", "mean active cmdCodes",
                 "importer-years DLR active (%)", "within-pair DLR switchers (%)",
                 "(i,j,t) with zero active categories (%)"),
  value = c(nrow(em), n_distinct(em$pair), round(mean(em$n_bpm6), 2),
            round(mean(em$n_cmd), 2), round(100 * mean(em$dlr_any_jt), 1),
            round(100 * mean(sw$v > 1), 1), round(100 * mean(em$n_bpm6 == 0), 2)),
  stringsAsFactors = FALSE)

model_stats <- data.frame(
  model        = names(models),
  estimator    = unname(est_type[names(models)]),
  outcome      = unname(outcome_lbl[names(models)]),
  fixed_effects = "exp_time + pair",
  cluster      = "pair_id",
  n_obs        = sapply(models, nobs),
  stringsAsFactors = FALSE)

readme <- data.frame(
  sheet = c("readme", "coefficients", "diagnostics", "model_stats"),
  description = c(
    "This sheet.",
    "DLR coefficient per model: estimate, SE, p, stars, % or pp effect, 95% CI, N.",
    "Country-level extensive-margin panel diagnostics and identifying variation.",
    "Per-model estimator, outcome, fixed effects, clustering, N."),
  stringsAsFactors = FALSE)

xlsx_path <- file.path(TABLES_DIR, "extensive_margin_country_results.xlsx")
writexl::write_xlsx(list(readme = readme, coefficients = coef_tbl,
                         diagnostics = diag_tbl, model_stats = model_stats), xlsx_path)
readr::write_csv(coef_tbl, file.path(TABLES_DIR, "extensive_margin_country_coefficients.csv"))
saveRDS(models, file.path(OUTPUT_DIR, "extensive_margin_country_models.rds"))
saveRDS(em,     file.path(OUTPUT_DIR, "extensive_margin_country_panel.rds"))
cat(sprintf("Saved workbook: %s\n", xlsx_path))
cat(sprintf("Saved CSV:      %s\n", file.path(TABLES_DIR, "extensive_margin_country_coefficients.csv")))
cat("Saved models + panel as .rds in gravity_results/.\n")

## 10 — Interpretation & limitations

**Reading the headline (E1).** $(\exp\hat\beta-1)\times100$ is the percent change in the number of active BPM6 service categories an exporter sells into an importer when that importer has an active data-localization regime. A negative, significant coefficient is the Melitz–Chaney extensive-margin prediction (higher $\varphi^{*}$ $\Rightarrow$ lower $M_{ij}$); E3/E4 add the dose-response in the number of restricted categories; E2/E4 re-run on the finer `cmdCode` varieties.

**Expected pattern.** In development the uncontrolled country-pair extensive-margin effects are near zero (slightly positive on `n_cmd` before importer controls, reflecting importer-size confounding that the GDP/population controls absorb). This is consistent with the thesis's standing finding that **trade is sticky on the extensive margin and DLR operates mainly through the intensive margin** — and it reframes, rather than contradicts, the title.

**Limitations to document.** (1) The paper is theory-only; the variety-count is a justified empirical bridge. (2) BPM6 varieties are coarse — `n_cmd` mitigates but services have no firm-level variety data. (3) Without intra-national services flows the Heid–Larch–Yotov (2021) domestic-trade identification of importer-level policies is infeasible, so inward multilateral resistance is only proxied by importer GDP/population — **the one choice worth raising with the supervisor.** (4) $\theta$ and $\sigma$ are not separately identified; the coefficient is the reduced-form extensive-margin semi-elasticity, interpreted through $\theta-\sigma+1$.